# Deep Learning Basics — Build Your First ANN (Artificial Neural Network)

## Project: Student Placement Prediction

**Goal:** Predict whether a student gets **Placed** or **Not Placed** based on their academic data.

### Features (X):
| Feature | Description |
|---------|------------|
| `study_hours_per_day` | Average daily study hours |
| `tenth_cgpa` | 10th standard CGPA (out of 10) |
| `twelfth_percentage` | 12th standard percentage |
| `college_cgpa` | College CGPA (out of 10) |
| `internships` | Number of internships done |
| `projects` | Number of projects completed |

### Target (y):
- `1` = Placed
- `0` = Not Placed

---

### What is an ANN?
An **Artificial Neural Network** is inspired by the human brain. It has:
- **Input Layer** → Takes in features (study hours, CGPA, etc.)
- **Hidden Layers** → Learns patterns from data
- **Output Layer** → Gives prediction (Placed / Not Placed)

```
      Input Layer   Hidden Layer 1     Hidden Layer 2     Output Layer
[study_hours] 6 ●   ──→  ●                  ●              
[tenth_cgpa]  8 ●   ──→  ●                  ●             
[twelfth_%]   7 ●   ──→  ●                  ●              
[college_cgpa]8 ●   ──→  ●                  ●                   ●     → [Placed?]
[internships] 2 ●   ──→  ●                  ●                                   
[projects]    2 ●   ──→  ●                  ●                                   
```

## Step 1: Import Libraries

We need:
- **numpy** → for numerical operations
- **pandas** → for data handling
- **tensorflow / keras** → to build our neural network
- **sklearn** → for data splitting and scaling

Tensorflow  + keras ---> Google   
                          
<!-- pytorch   ---> facebook -->

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TensorFlow & Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("TensorFlow version:", tf.__version__)

## Step 2: Create the Dataset

Since we don't have a real dataset, let's **create one** with realistic student data.

**Logic:** Students with higher CGPA, more study hours, internships, and projects are more likely to get placed.

In [ ]:
np.random.seed(42)
n_samples = 500

# Generate features
study_hours = np.round(np.random.uniform(1, 10, n_samples), 1)
tenth_cgpa = np.round(np.random.uniform(4, 10, n_samples), 1)
twelfth_pct = np.round(np.random.uniform(40, 98, n_samples), 1)
college_cgpa = np.round(np.random.uniform(4, 10, n_samples), 1)
internships = np.random.randint(0, 5, n_samples)
projects = np.random.randint(0, 6, n_samples)

# Create a "placement score" — higher = more likely placed
score = (
    study_hours * 0.15 +
    tenth_cgpa * 0.15 +
    twelfth_pct * 0.02 +
    college_cgpa * 0.25 +
    internships * 0.5 +
    projects * 0.4 +
    np.random.normal(0, 0.5, n_samples)  # some randomness
)

# Convert to binary: 1 = Placed, 0 = Not Placed
threshold = np.median(score)
placed = (score >= threshold).astype(int)

# Create DataFrame
df = pd.DataFrame({
    'study_hours_per_day': study_hours,
    'tenth_cgpa': tenth_cgpa,
    'twelfth_percentage': twelfth_pct,
    'college_cgpa': college_cgpa,
    'internships': internships,
    'projects': projects,
    'placed': placed
})

print(f"Dataset shape: {df.shape}")
print(f"\nPlacement distribution:\n{df['placed'].value_counts()}")
df.head(10)

## Step 3: Prepare the Data (X and y)

**Very Important for beginners:**
- `X` = All input features (what the model sees)
- `y` = Target/label (what the model predicts)

We also **split** the data into:
- **Training set (80%)** → Model learns from this
- **Test set (20%)** → We check if model actually learned

In [ ]:
# Separate features (X) and target (y)
X = df.drop('placed', axis=1)   # everything except 'placed'
y = df['placed']                 # only 'placed' column

print("X shape:", X.shape)  # (500, 6) → 500 students, 6 features
print("y shape:", y.shape)  # (500,)   → 500 labels

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

## Step 4: Feature Scaling (Very Important!)

**Why scale?**
- Features like `twelfth_percentage` (40–98) and `internships` (0–4) have very different ranges
- Neural networks learn **much faster** when all features are on the same scale
- We use **StandardScaler** → transforms data to mean=0, std=1

```
Before scaling:  study_hours = 8.0,  twelfth_pct = 92.0,  internships = 3
After scaling:   study_hours = 0.72, twelfth_pct = 0.85,  internships = 0.94
```

> **Rule:** Always fit scaler on TRAIN data only, then transform both train and test!

In [ ]:
# Scale the features
scaler = StandardScaler()

# Fit on train data ONLY, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)    # ← only transform, NOT fit!

# Let's see the difference
print("Before Scaling (first row of X_train):")
print(X_train.iloc[0].values)

print("\nAfter Scaling (first row of X_train_scaled):")
print(X_train_scaled[0].round(3))

## Step 5: Build the ANN Model

Now the exciting part! We build a **Sequential** model layer by layer.

### Understanding Each Layer:

```
Layer              Neurons   Activation   Why?
─────────────────────────────────────────────────────
Input Layer        6         —            One neuron per feature
Hidden Layer 1     16        ReLU         Learns simple patterns
Hidden Layer 2     8         ReLU         Learns complex patterns
Output Layer       1         Sigmoid      Outputs probability (0 to 1)
```

### What is an Activation Function?
- **ReLU (Rectified Linear Unit):** `if x > 0 → x, else → 0` — used in hidden layers
- **Sigmoid:** Squishes output between 0 and 1 — perfect for binary classification

```
ReLU:     ___/        Sigmoid:    __/‾‾
                                /
```

In [ ]:
# Build the ANN step by step
model = Sequential()

# Hidden Layer 1: 16 neurons, ReLU activation
# input_dim=6 tells Keras we have 6 input features
model.add(Dense(units=16, activation='relu', input_dim=6))

# Hidden Layer 2: 8 neurons, ReLU activation
model.add(Dense(units=8, activation='relu'))

# Output Layer: 1 neuron, Sigmoid activation (for binary: placed/not placed)
model.add(Dense(units=1, activation='sigmoid'))

# Let's see what our model looks like
model.summary()

## 📘 Deep Dive: `input_dim`, `units`, and Activation Functions (Beginner Guide)

### What is `input_dim`?

`input_dim=6` tells the model: *"each student has 6 input numbers."* Our 6 features are:

| # | Feature | Example value |
|---|---------|---------------|
| 1 | `study_hours_per_day` | 8.0 |
| 2 | `tenth_cgpa` | 9.1 |
| 3 | `twelfth_percentage` | 92.0 |
| 4 | `college_cgpa` | 8.6 |
| 5 | `internships` | 3 |
| 6 | `projects` | 4 |

**Rule:** `input_dim` must always match the number of columns in `X` (check with `X_train.shape`).

---

### What is `units`?

`units=16` means the layer has **16 neurons**. Each neuron tries to learn a different pattern from the data (one neuron might focus on "good academic performance", another on "hands-on experience"). We don't assign these tasks manually — the network figures them out while training.

```
Input (6 features) → Hidden Layer 1 (16 neurons) → Hidden Layer 2 (8 neurons) → Output (1 neuron)
```

The output layer uses `units=1` because we only need **one number**: the probability of being placed.

---

### What is an Activation Function?

An activation function is a **decision gate** inside each neuron — it decides how much signal to pass forward. Without activation functions, a neural network could only learn straight-line relationships, no matter how many layers it had.

#### ReLU (used in hidden layers)

`relu` keeps positive numbers as they are, and turns negative numbers into 0:

| Input | ReLU output |
|------:|------------:|
| 10 | 10 |
| 3 | 3 |
| 0 | 0 |
| -2 | 0 |
| -15 | 0 |

ReLU is a good default for hidden layers because it's simple, fast, and works well in practice.

#### Sigmoid (used in the output layer)

`sigmoid` squishes any number into a value **between 0 and 1**, so we can read it as a probability.

```
Model output = 0.87  →  "87% chance this student gets placed"
```

We turn that probability into a final answer like this:

```python
if probability >= 0.5:
    print("Placed")
else:
    print("Not placed")
```

We use sigmoid here because this is a **binary classification** problem (Placed = 1, Not Placed = 0).

---

### Other Common Activation Functions (good to know)

| Activation | Used for | Simple meaning |
|---|---|---|
| `relu` | Hidden layers | Negative → 0, positive stays the same |
| `sigmoid` | Binary output (2 classes) | Squishes value between 0 and 1 |
| `softmax` | Multi-class output (3+ classes) | Gives a probability for every class, all adding up to 1 |
| `tanh` | Some hidden layers | Squishes value between -1 and 1 |
| `linear` | Regression output (predicting a number) | No squishing — returns any number |

**Example — predicting a student's department (multi-class):**
```python
Dense(units=3, activation='softmax')
# Output: CS = 0.70, Mechanical = 0.20, Civil = 0.10 → model picks "CS"
```

**Example — predicting salary (regression):**
```python
Dense(units=1, activation='linear')
# Output: 45000 (not restricted to 0–1)
```

## Step 6: Compile the Model

Before training, we need to tell the model **HOW to learn**:

| Setting | What it does | Our choice |
|---------|-------------|------------|
| **Optimizer** | How the model updates weights | `adam` (best general-purpose) |
| **Loss Function** | How to measure error | `binary_crossentropy` (for yes/no problems) |
| **Metrics** | What to track during training | `accuracy` |

### How does training work?
```
1. Model sees data → makes a prediction
2. Loss function checks: "How wrong was it?"
3. Optimizer adjusts weights to reduce the error
4. Repeat for many epochs (rounds)!
```

In [ ]:
# Compile the model
model.compile(
    optimizer='adam',                    # Adam optimizer (adaptive learning rate)
    loss='binary_crossentropy',          # Binary because: placed (1) or not (0)
    metrics=['accuracy']                 # Track accuracy during training
)

print("Model compiled successfully!")

## 📘 Deep Dive: Optimizer, Loss Function, and Metrics (Beginner Guide)

### What is a Loss Function?

The loss function tells the model **how wrong** its prediction was — think of it as a "mistake score". Small loss = good prediction, large loss = bad prediction.

```
Correct answer = 1 (Placed)
Prediction = 0.95  →  small loss (good!)
Prediction = 0.10  →  large loss (bad!)
```

During training, the network keeps adjusting itself to make this loss smaller.

**Why `binary_crossentropy` for us?** Because we have exactly two classes (Placed/Not Placed), and our output layer uses `sigmoid`, which naturally pairs with this loss.

| Problem type | Output activation | Loss function |
|---|---|---|
| Binary classification (2 classes) | `sigmoid` | `binary_crossentropy` |
| Multi-class, one-hot labels | `softmax` | `categorical_crossentropy` |
| Multi-class, integer labels (0,1,2,...) | `softmax` | `sparse_categorical_crossentropy` |
| Regression (predicting a number) | `linear` | `mse` or `mae` |

---

### What is an Optimizer?

The optimizer is the strategy the model uses to **fix its mistakes** and improve. A simple training cycle:

```
1. Model makes a prediction
2. Loss function measures the mistake
3. Optimizer nudges the model's internal weights to reduce that mistake
4. Repeat, many times (epochs)
```

**Why `adam`?** It's a great all-round default — it automatically adjusts how big each correction should be, so it usually learns faster and needs less manual tuning than simpler optimizers.

| Optimizer | Simple description |
|---|---|
| `adam` | Good general-purpose choice, adjusts itself automatically |
| `sgd` | Basic optimizer, often needs manual tuning |
| `rmsprop` | Popular for sequence/time-series problems |

**Learning rate** controls how big each correction is (like step size while walking downhill toward the lowest point):
```python
from tensorflow.keras.optimizers import Adam
optimizer = Adam(learning_rate=0.001)   # 0.001 is a common starting value
```

---

### What are Metrics?

Metrics are numbers shown to **us (humans)** so we can understand how well the model is doing. They don't directly control learning — the optimizer only reacts to the loss.

- **Accuracy** — out of all predictions, how many were correct?
  ```
  85 correct out of 100 students → accuracy = 85%
  ```
- **Precision** — out of students predicted as "placed", how many were actually placed? Important when false positives are costly.
- **Recall** — out of students who were actually placed, how many did the model correctly find? Important when missing positives is costly.
- **AUC** — measures how well the model separates the two classes overall (0.5 = random guessing, 0.9+ = excellent).

```python
from tensorflow.keras.metrics import Precision, Recall, AUC

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall'), AUC(name='auc')]
)
```

## Step 7: Train the Model

Now we **fit** the model on our training data!

Key terms:
- **epochs** = Number of times the model sees the ENTIRE training data (50 rounds)
- **batch_size** = How many samples the model processes before updating weights (32 at a time)
- **validation_split** = Use 20% of training data to check progress during training

```
Epoch 1:  Model is guessing randomly          → accuracy ~50%
Epoch 10: Model starts learning patterns       → accuracy ~70%
Epoch 50: Model has learned well               → accuracy ~85%+
```

In [ ]:
# Train the model!
history = model.fit(
    X_train_scaled,          # Scaled training features
    y_train,                 # Training labels
    epochs=50,               # 50 rounds of training
    batch_size=32,           # Process 32 samples at a time
    validation_split=0.2,    # Use 20% of train data for validation
    verbose=1                # Show progress
)

print("\nTraining complete!")

## Step 8: Visualize Training Progress

Let's plot how the model improved over each epoch.
- **Loss** should go DOWN (model making fewer mistakes)
- **Accuracy** should go UP (model getting better)

If **validation loss goes UP** while training loss goes down → **Overfitting!**

In [ ]:
# Plot Training History
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Loss ---
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Model Loss Over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Plot 2: Accuracy ---
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('Model Accuracy Over Epochs', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 9: Evaluate on Test Data

Now the real test! The model has **never seen** the test data before.
This tells us how well the model will perform on new, unseen students.

In [ ]:
# Evaluate on test data
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print("=" * 40)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.1f}%)")
print("=" * 40)

## Step 10: Confusion Matrix & Classification Report

Let's see exactly WHERE the model is getting things right or wrong.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Get predictions (sigmoid outputs probabilities, so we threshold at 0.5)
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Visualize
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Placed', 'Placed'],
            yticklabels=['Not Placed', 'Placed'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Student Placement')
plt.tight_layout()
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Placed', 'Placed']))

## Step 11: Make Predictions on New Students!

Let's test our model with **brand new student data** it has never seen before.

In [ ]:
# Create new student data
new_students = pd.DataFrame({
    'study_hours_per_day': [8.0, 2.0, 6.5],
    'tenth_cgpa':          [9.2, 5.5, 7.8],
    'twelfth_percentage':  [92.0, 55.0, 78.0],
    'college_cgpa':        [8.5, 5.0, 7.2],
    'internships':         [3, 0, 2],
    'projects':            [4, 1, 3]
})

# IMPORTANT: Scale the new data using the SAME scaler
new_students_scaled = scaler.transform(new_students)

# Predict
predictions = model.predict(new_students_scaled)

print("=" * 60)
print("PREDICTIONS FOR NEW STUDENTS")
print("=" * 60)

for i, (_, student) in enumerate(new_students.iterrows()):
    prob = predictions[i][0]
    status = "PLACED ✓" if prob > 0.5 else "NOT PLACED ✗"
    print(f"\nStudent {i+1}:")
    print(f"  Study Hours: {student['study_hours_per_day']}h | "
          f"10th CGPA: {student['tenth_cgpa']} | "
          f"12th%: {student['twelfth_percentage']}%")
    print(f"  College CGPA: {student['college_cgpa']} | "
          f"Internships: {int(student['internships'])} | "
          f"Projects: {int(student['projects'])}")
    print(f"  Prediction: {status} (Probability: {prob:.2%})")

## Step 12: Save the Model (So you don't have to retrain!)

Once trained, save the model to reuse later without retraining.

In [ ]:
# Save the model
model.save('student_placement_ann.keras')
print("Model saved as 'student_placement_ann.keras'")

# To load later:
# loaded_model = keras.models.load_model('student_placement_ann.keras')
# loaded_model.predict(new_data_scaled)

---

## Quick Recap — What We Did

| Step | What | Code |
|------|------|------|
| 1 | Import libraries | `import tensorflow as tf` |
| 2 | Create dataset | `pd.DataFrame(...)` |
| 3 | Split data | `train_test_split(X, y)` |
| 4 | Scale features | `StandardScaler().fit_transform()` |
| 5 | Build ANN | `Sequential() → Dense(16, relu) → Dense(8, relu) → Dense(1, sigmoid)` |
| 6 | Compile | `model.compile(optimizer='adam', loss='binary_crossentropy')` |
| 7 | Train | `model.fit(X_train, y_train, epochs=50)` |
| 8 | Visualize | Plot loss & accuracy curves |
| 9 | Evaluate | `model.evaluate(X_test, y_test)` |
| 10 | Metrics | Confusion matrix & classification report |
| 11 | Predict | `model.predict(new_data)` |
| 12 | Save | `model.save('model.keras')` |

### Key Concepts to Remember:
- **ReLU** → Hidden layers | **Sigmoid** → Output layer (binary)
- **binary_crossentropy** → For 2-class problems
- **Adam** → Best default optimizer
- **Always scale** your features before feeding to ANN
- **Watch for overfitting** → validation loss increasing while training loss decreasing